# C03 — Transfer Learning in PyTorch

> **Audience**: PhD students · **Framework**: PyTorch · **Dataset**: CIFAR-10

Transfer learning is the dominant paradigm in applied deep learning: start from
weights pretrained on a large dataset, adapt to your target task with far less data.

**Two strategies covered**

| Strategy | When to use | How |
|----------|-------------|-----|
| Feature extraction | Very small dataset (<1 000 samples) | Freeze all backbone layers; train only the head |
| Fine-tuning | Moderate dataset; target domain differs from ImageNet | Unfreeze some/all layers; use a very low LR |

**Third strategy (bonus)**: use the frozen backbone as a feature extractor
and train a classical ML classifier (logistic regression) on the extracted features.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# 1) Dataset — CIFAR-10

CIFAR-10: 60 000 colour 32×32 images, 10 classes (plane, car, bird, cat, deer,
dog, frog, horse, ship, truck).

**ImageNet normalisation**
All torchvision pretrained models expect inputs normalised with ImageNet
statistics. This is because the backbone weights were calibrated to this
particular input distribution. Using different statistics degrades performance.

```
mean = [0.485, 0.456, 0.406]   # per-channel mean of ImageNet
std  = [0.229, 0.224, 0.225]   # per-channel std  of ImageNet
```

**Resize to 224×224**
ImageNet models expect 224×224 inputs. CIFAR-10 is only 32×32, so we must
resize. This is a domain gap we accept; in practice you would use a model
pretrained on a closer domain when available.

In [ ]:
CIFAR_CLASSES = [
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# ImageNet normalisation — required for all torchvision pretrained models
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Resize 32×32 → 224×224 for ImageNet-pretrained backbones
transfer_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_aug_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=16),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_train = torchvision.datasets.CIFAR10(
    root="./data", train=True,  download=True, transform=train_aug_transform
)
full_val   = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transfer_transform
)

# Use a 2 000-sample subset to simulate low-data regime (feature extraction use-case)
torch.manual_seed(42)
subset_idx = torch.randperm(len(full_train))[:2000].tolist()
small_train = Subset(full_train, subset_idx)

train_loader_small = DataLoader(small_train, batch_size=64,  shuffle=True,  num_workers=2)
train_loader_full  = DataLoader(full_train,  batch_size=128, shuffle=True,  num_workers=2)
val_loader         = DataLoader(full_val,    batch_size=256, shuffle=False, num_workers=2)

print(f"Small train : {len(small_train)} samples")
print(f"Full  train : {len(full_train)} samples")
print(f"Val         : {len(full_val)} samples")

# 2) Loading a Pretrained Backbone

`torchvision.models` provides architectures with weights pretrained on ImageNet-1K.

**Modern weights API** (torchvision ≥ 0.13)
```python
# Old (deprecated): pretrained=True
# New (correct)   : weights=models.ResNet18_Weights.DEFAULT
```
The new API explicitly names the weight set, making experiments reproducible
as model variants and weight versions multiply.

**Inspecting the architecture**
It is essential to understand which layers your backbone has before modifying it.
For ResNet: the final layer is `model.fc` (fully connected).
For MobileNetV2: the final layer is `model.classifier[1]` (inside a Sequential).

In [ ]:
# Load ResNet-18 with the best available ImageNet weights
backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Inspect the final classification layer
print("ResNet18 final layer:")
print(backbone.fc)          # Linear(in_features=512, out_features=1000, bias=True)
print(f"  in_features : {backbone.fc.in_features}")
print(f"  out_features: {backbone.fc.out_features}  (ImageNet 1000 classes)")

# Count parameters before modification
total_params  = sum(p.numel() for p in backbone.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Visualise sample predictions with the unmodified pretrained model
backbone.eval().to(DEVICE)

sample_images, sample_labels = next(iter(val_loader))
with torch.no_grad():
    # (batch_num, 3, 224, 224) → (batch_num, 1000)
    imagenet_logits = backbone(sample_images[:4].to(DEVICE))
    imagenet_preds  = imagenet_logits.argmax(1).cpu().tolist()

print(f"\nImageNet pred indices (not CIFAR labels): {imagenet_preds}")

# 3) Strategy 1 — Feature Extraction

**Pattern**: freeze the entire backbone, replace only the final classification head.

```
Frozen backbone (ImageNet weights, not updated)
  ↓
New head: Linear(backbone_dim → n_target_classes)  ← only this is trained
```

**When to use**
- Very small target dataset (hundreds to low thousands of samples)
- Sufficient domain similarity to ImageNet (natural images)
- Time / compute constraints

**Critical detail — `requires_grad = False`**
Setting `requires_grad=False` on a parameter tells autograd to skip computing
its gradient during `backward()`. This both saves memory and prevents the
backbone from moving away from its pretrained values.

In [ ]:
def make_feature_extractor(n_classes: int = 10) -> nn.Module:
    """
    Loads ResNet-18, freezes all layers, replaces the classification head.

    Args:
        n_classes : Number of target classes

    Returns:
        model : ResNet-18 with frozen backbone and new linear head
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Freeze every parameter in the backbone — no gradient computation
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final FC layer with a new one for our n_classes
    # Only this layer will be trained (requires_grad=True by default for new modules)
    # (batch_num, 512) → (batch_num, n_classes)
    in_features   = model.fc.in_features   # 512 for ResNet-18
    model.fc      = nn.Linear(in_features, n_classes)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")

    return model


def train_loop(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_epochs: int,
    lr: float,
) -> dict:
    """Training loop returning history dict."""
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    history   = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, n_epochs + 1):
        # ── Train ──
        model.train()
        running_loss = 0.0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            # (batch_num, 3, 224, 224) → (batch_num, n_classes)
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X.size(0)
        scheduler.step()

        # ── Validate ──
        model.eval()
        val_loss, correct = 0.0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                logits = model(X)
                val_loss += criterion(logits, y).item() * X.size(0)
                correct  += (logits.argmax(1) == y).sum().item()

        n     = len(val_loader.dataset)
        tl    = running_loss / len(train_loader.dataset)
        vl    = val_loss / n
        vacc  = correct / n
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_acc"].append(vacc)

        print(f"  Epoch {epoch:02d}/{n_epochs} | train_loss={tl:.4f} "
              f"val_loss={vl:.4f} val_acc={vacc:.3f}")

    return history


# Train feature extractor on small dataset (2 000 samples)
print("=== Feature Extraction (2 000 samples) ===")
feat_model = make_feature_extractor(n_classes=10)
history_feat = train_loop(feat_model, train_loader_small, val_loader, n_epochs=5, lr=1e-3)

# 4) Strategy 2 — Fine-Tuning

**Pattern**: unfreeze all (or partial) layers and train with a very low learning rate.

```
Pretrained backbone (initially frozen, then released) ← tiny LR keeps these stable
  ↓
New head: Linear(backbone_dim → n_target_classes)     ← trained with normal LR
```

**Two-phase recipe** (best practice)
1. **Phase 1**: train only the new head for a few epochs (warm-up).
   This prevents the large initial gradients from the random head from
   damaging the pretrained backbone before the head is sensible.
2. **Phase 2**: unfreeze all layers, set a very low LR (typically 10–100×
   smaller than Phase 1), and continue training.

**Differential learning rates** (advanced)
For even better results, assign layer-group-specific LRs:
- Earlier layers (generic features): smallest LR
- Later layers (task-specific): larger LR
- New head: largest LR

This reflects the intuition that early layers need minimal adjustment
while later layers must adapt more to the new domain.

In [ ]:
def make_finetune_model(n_classes: int = 10) -> nn.Module:
    """
    Loads ResNet-18 and prepares it for two-phase fine-tuning.

    Phase 1: only the new head is trainable.
    Phase 2: all layers are made trainable (call unfreeze_backbone()).
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Freeze backbone for Phase 1
    for param in model.parameters():
        param.requires_grad = False

    # (batch_num, 512) → (batch_num, n_classes)
    model.fc = nn.Linear(model.fc.in_features, n_classes)

    return model


def unfreeze_backbone(model: nn.Module) -> None:
    """
    Unfreezes all backbone parameters for Phase 2 fine-tuning.
    The head is already trainable from Phase 1.
    """
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"All {trainable:,} parameters now trainable")


def make_param_groups(model: nn.Module, lr_backbone: float, lr_head: float) -> list:
    """
    Creates parameter groups with different learning rates.

    Backbone (pretrained) gets a lower LR to preserve learned features.
    Head (new) gets a higher LR to learn the task quickly.

    Args:
        lr_backbone : Learning rate for the pretrained backbone layers
        lr_head     : Learning rate for the new classification head

    Returns:
        param_groups : List suitable for torch.optim constructors
    """
    head_params     = list(model.fc.parameters())
    head_ids        = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

    return [
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params,     "lr": lr_head},
    ]


# ── Phase 1: warm up the head (3 epochs, backbone frozen) ─────────────────────
print("=== Phase 1: head warm-up ===")
ft_model = make_finetune_model(n_classes=10).to(DEVICE)
history_ft1 = train_loop(ft_model, train_loader_full, val_loader, n_epochs=3, lr=1e-3)

# ── Phase 2: fine-tune entire network with differential LRs ────────────────────
print("\n=== Phase 2: full fine-tune ===")
unfreeze_backbone(ft_model)

criterion_ft = nn.CrossEntropyLoss()
param_groups = make_param_groups(ft_model, lr_backbone=1e-5, lr_head=1e-4)
optimizer_ft = optim.AdamW(param_groups, weight_decay=1e-4)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=3)

history_ft2 = {"train_loss": [], "val_loss": [], "val_acc": []}
for epoch in range(1, 4):
    ft_model.train()
    running = 0.0
    for X, y in train_loader_full:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer_ft.zero_grad()
        loss = criterion_ft(ft_model(X), y)
        loss.backward()
        optimizer_ft.step()
        running += loss.item() * X.size(0)
    scheduler_ft.step()

    ft_model.eval()
    vl, correct = 0.0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            logits = ft_model(X)
            vl      += criterion_ft(logits, y).item() * X.size(0)
            correct += (logits.argmax(1) == y).sum().item()

    n    = len(val_loader.dataset)
    tl   = running / len(train_loader_full.dataset)
    vl  /= n
    vacc = correct / n
    history_ft2["train_loss"].append(tl)
    history_ft2["val_loss"].append(vl)
    history_ft2["val_acc"].append(vacc)
    print(f"  Epoch {epoch}/3 | train_loss={tl:.4f} val_loss={vl:.4f} val_acc={vacc:.3f}")

# 5) Strategy 3 — Feature Extraction + Classical ML Classifier

**Pattern**: use the frozen pretrained backbone as a fixed feature extractor,
run all training images through it *once*, cache the feature vectors, then
train a classical ML model (logistic regression, SVM, k-NN) on those vectors.

**Advantages**
- Faster iteration: no GPU required once features are extracted.
- Interpretable linear classifier on top of rich features.
- Useful diagnostic: if logistic regression on top of ResNet features already
  achieves 90%+ accuracy, you likely do not need to fine-tune at all.

**Implementation pattern**
```
for batch in loader:
    features = backbone(batch)    # forward through frozen backbone
    cache features and labels
sklearn_model.fit(all_features, all_labels)
```

In [ ]:
@torch.no_grad()
def extract_features(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> tuple:
    """
    Passes all data through the backbone and returns cached feature vectors.

    The backbone's FC layer is temporarily replaced with an identity to
    extract the penultimate feature representation.

    Args:
        model  : Pretrained model; fc layer will be bypassed
        loader : DataLoader to extract features from

    Returns:
        features : (num_samples, feature_dim) float32 array
        labels   : (num_samples,) int array
    """
    model.eval().to(device)

    # Swap the final layer for an identity to get raw feature vectors
    original_fc = model.fc
    model.fc    = nn.Identity()

    all_feats, all_labels = [], []

    for X, y in loader:
        X = X.to(device)
        # (batch_num, 3, 224, 224) → (batch_num, feature_dim=512)
        feats = model(X).cpu().numpy()
        all_feats.append(feats)
        all_labels.append(y.numpy())

    # Restore original head
    model.fc = original_fc

    return np.vstack(all_feats), np.concatenate(all_labels)


# Extract features from a fresh frozen ResNet-18
feat_extractor = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
feat_extractor.eval()

print("Extracting training features...")
X_train_feats, y_train = extract_features(feat_extractor, train_loader_full, DEVICE)
print("Extracting validation features...")
X_val_feats,   y_val   = extract_features(feat_extractor, val_loader, DEVICE)

print(f"Train features shape : {X_train_feats.shape}")  # (50000, 512)
print(f"Val   features shape : {X_val_feats.shape}")    # (10000, 512)

# Standardise features — important for logistic regression convergence
scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train_feats)
X_val_sc     = scaler.transform(X_val_feats)

# Train logistic regression on the cached features
print("\nFitting logistic regression on ResNet features...")
logreg = LogisticRegression(max_iter=500, C=1.0, solver="lbfgs", multi_class="multinomial")
logreg.fit(X_train_sc, y_train)

val_preds = logreg.predict(X_val_sc)
acc       = accuracy_score(y_val, val_preds)
print(f"Logistic regression on ResNet-18 features: val_acc = {acc:.3f}")

# 6) Strategy Comparison

**Summary of results** (approximate, varies by run)

| Strategy | Data used | Epochs | Expected val acc |
|---|---|---|---|
| Feature extraction | 2 000 samples | 5 | ~70–75% |
| Logistic reg on features | 50 000 samples | N/A | ~80–85% |
| Fine-tuning (Phase 1+2) | 50 000 samples | 6 | ~87–91% |

**Takeaways for research design**
- If your dataset has < 1 000 samples: use feature extraction or frozen features + SVM.
- If you have a few thousand samples: two-phase fine-tuning with differential LRs.
- Always start with a frozen backbone warm-up — it prevents the random head from
  corrupting pretrained weights before it learns sensible representations.

In [ ]:
# Consolidate validation accuracies
print("Strategy summary:")
print(f"  Feature extraction (2k samples) : {history_feat['val_acc'][-1]:.3f}")
print(f"  LogReg on ResNet features (50k) : {acc:.3f}")
print(f"  Fine-tuning Phase 1 (50k)       : {history_ft1['val_acc'][-1]:.3f}")
print(f"  Fine-tuning Phase 2 (50k)       : {history_ft2['val_acc'][-1]:.3f}")

# Plot all val accuracy curves together
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_feat["val_acc"],                   label="Feature extraction (2k)", marker="o")
ax.plot([acc] * 6,                                 label="LogReg on features (50k)", linestyle="--")
ax.plot(history_ft1["val_acc"],                    label="Fine-tune Ph.1 (50k)", marker="s")
ax.plot(range(3, 6), history_ft2["val_acc"],       label="Fine-tune Ph.2 (50k)", marker="^")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_title("Transfer learning strategy comparison — CIFAR-10")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

# Summary

**The transfer learning decision tree**

```
Is your target domain close to ImageNet natural images?
├── Yes → use pretrained backbone
│   ├── Very small data (<1 000)  → frozen features + classical ML
│   ├── Small data (1–10k)        → feature extraction, train only head
│   └── Moderate data (>10k)      → two-phase fine-tuning, differential LRs
└── No  → train from scratch or use domain-specific pretrained weights
```

**Key implementation checklist**
1. Use `weights=...Weights.DEFAULT` (not deprecated `pretrained=True`)
2. Set `requires_grad=False` on backbone before Phase 1
3. Use `filter(lambda p: p.requires_grad, model.parameters())` in the optimiser
4. Call `model.eval()` during validation — BatchNorm / Dropout behave differently
5. Use `optimizer.zero_grad()` before every backward pass